# Chapter 6 — Exercises: Worked Solutions

Worked solutions for Chapter 6 (Pure Component Parameters).

**Exercises:**
1. Parameter regression for a 2B fluid from vapor pressure data
2. Association scheme selection for different compound families
3. Parameter landscape exploration

In [1]:
import importlib, subprocess, sys
try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False); ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
except Exception:
    try: import neqsim
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
print(f"NeqSim mode: {NEQSIM_MODE}")

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim\src\main\resources
  3. C:\Users\ESOL\.m2\repository\com\h2database\h2\2.4.240\h2-2.4.240.jar
  4. C:\Users\ESOL\.m2\repository\org\apache\logging\log4j\log4j-api\2.25.4\log4j-api-2.25.4.jar
  5. C:\Users\ESOL\.m2\repository\org\apache\logging\log4j\log4j-core\2.25.4\log4j-core-2.25.4.jar
  6. C:\Users\ESOL\.m2\repository\com\thoughtworks\xstream\xstream\1.4.21\xstream-1.4.21.jar
  7. C:\Users\ESOL\.m2\repository\io\github\x-stream\mxparser\1.2.2\mxparser-1.2.2.jar
  8. C:\Users\ESOL\.m2\repository\xmlpull\xmlpull\1.1.3.1\xmlpull-1.1.3.1.jar
  9. C:\Users\ESOL\.m2\repository\org\apache\commons\commons-lang3\3.20.0\commons-lang3-3.20.0.jar
  10. C:\Users\ESOL\.m2\repository\org\apache\commons\commons-math3\3.6.1\commons-math3-3.6.1.jar
  11. C:\Users\ESOL\.m2\repository\org\ejml\ejml-all\0.44.0\ejml-all-0.44.0.jar
  12


JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim mode: devtools


In [2]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PALETTE = ["#2171b5", "#e6550d", "#31a354", "#756bb1", "#e7298a", "#66a61e"]
BLUE, ORANGE, GREEN, PURPLE, PINK, LIME = PALETTE
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.labelsize": 10, "axes.titlesize": 10, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "xtick.direction": "in", "ytick.direction": "in", "axes.linewidth": 0.6, "lines.linewidth": 1.2, "grid.linewidth": 0.3, "grid.alpha": 0.4, "savefig.dpi": 300, "figure.dpi": 150})
FIGURES_DIR = Path("../figures"); FIGURES_DIR.mkdir(exist_ok=True)
def save(fig, name): fig.savefig(str(FIGURES_DIR / name), dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig); print(f"Saved: {name}")
R = 8.314

---
## Exercise 6.1 — Parameter Regression Concept

**Problem:** For a 2B associating fluid with known vapor pressure data,
describe the regression procedure and compute the objective function
landscape for $\varepsilon/R$ and $\beta$.

The objective function is:

$$F(\varepsilon, \beta) = \sum_i \left(\frac{P_{\text{calc},i} - P_{\text{exp},i}}{P_{\text{exp},i}}\right)^2$$

### Solution

We use a simplified analytical model to illustrate the regression landscape.
In practice, NeqSim's parameter fitting tools handle the full EoS regression.

In [3]:
# Simplified: model vapor pressure as function of association params
# P_vap ~ P_vap_srk * exp(-n_sites * (1/XA - 1) * eps/(2*R*T))
# where XA depends on eps, beta

T_ref = 373.15  # 100 C
rho_liq = 55000.0  # mol/m3 (water liquid)
b_val = 1.45e-5
P_vap_srk = 2.5  # bar (SRK prediction without association)
P_vap_exp = 1.013  # bar (experimental at 100 C)

eps_range = np.linspace(500, 3500, 60)  # eps/R in K
beta_range = np.linspace(0.01, 0.15, 50)

EPS, BETA = np.meshgrid(eps_range, beta_range)
OBJ = np.zeros_like(EPS)

for i in range(len(beta_range)):
    for j in range(len(eps_range)):
        eps_R = eps_range[j]
        beta = beta_range[i]
        BF = np.exp(eps_R / T_ref) - 1.0
        eta = b_val * rho_liq / 4.0
        g = 1.0 / (1 - 1.9 * eta)
        Delta = g * BF * b_val * beta
        XA = (-1 + np.sqrt(1 + 8 * rho_liq * Delta)) / (4 * rho_liq * Delta)
        # Simplified vapor pressure model
        P_calc = P_vap_srk * np.exp(-4 * (1/XA - 1) * eps_R / (2 * T_ref))
        OBJ[i, j] = ((P_calc - P_vap_exp) / P_vap_exp)**2

fig, ax = plt.subplots(figsize=(4.0, 3.0))
cs = ax.contourf(EPS, BETA, np.log10(OBJ + 1e-10), levels=20, cmap="viridis")
ax.contour(EPS, BETA, np.log10(OBJ + 1e-10), levels=[-3, -2, -1, 0], colors="white",
           linewidths=0.5, linestyles="--")
ax.plot(2003.1, 0.0692, 'r*', markersize=10, label="Literature (water)")
fig.colorbar(cs, ax=ax, label=r"$\log_{10}(F)$", shrink=0.85)
ax.set_xlabel(r"$\varepsilon / R$ (K)"); ax.set_ylabel(r"$\beta$")
ax.set_title("Exercise 6.1: Objective function landscape")
ax.legend(frameon=True, fontsize=7)
fig.tight_layout()
save(fig, "fig_ch06_ex01_regression_landscape.png")

Saved: fig_ch06_ex01_regression_landscape.png


**Answer:** The objective function landscape reveals a valley-like minimum
in ($\varepsilon/R$, $\beta$) space. Multiple parameter combinations can
give similar vapor pressures, which is why simultaneous fitting to vapor
pressure AND liquid density is essential to obtain unique, physically
meaningful parameters.

---
## Exercise 6.2 — Association Scheme Selection

**Problem:** Assign the appropriate association scheme for each compound:

| Compound | Scheme | Sites | Reason |
|----------|--------|-------|--------|
| Water | 4C | 4 (2H + 2e) | Two OH groups |
| Methanol | 2B | 2 (1H + 1e) | One OH group |
| Acetic acid | 1A | 1 | Dimerization |
| Ethane | Non-assoc. | 0 | No H-bonding |
| Ammonia | 3B | 3 (1e + 2H) | Or 4C |

### Solution

In [4]:
# Verify with NeqSim: CPA vapor pressures for different compounds
if NEQSIM_MODE == "pip":
    SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
else:
    SystemSrkCPAstatoil = ns.SystemSrkCPAstatoil
    ThermodynamicOperations = ns.ThermodynamicOperations

compounds = ["water", "methanol", "ethanol", "MEG"]
T_test = 273.15 + 80.0

print(f"CPA vapor pressures at 80 C:")
print(f"{'Compound':<12} {'P_vap (bar)':<12}")
print("-" * 24)
for comp in compounds:
    try:
        f = SystemSrkCPAstatoil(T_test, 1.0)
        f.addComponent(comp, 1.0)
        f.setMixingRule(10)
        ops = ThermodynamicOperations(f)
        ops.bubblePointPressureFlash(False)
        Pvap = float(f.getPressure("bara"))
        print(f"{comp:<12} {Pvap:<12.4f}")
    except Exception as e:
        print(f"{comp:<12} {'Error':<12}")

CPA vapor pressures at 80 C:
Compound     P_vap (bar) 
------------------------


water        0.4682      
methanol     1.8099      
ethanol      1.1016      
MEG          0.0068      


**Answer:** The association scheme determines the number and types of
bonding sites. The 4C scheme for water (4 sites: 2 proton donors + 2
lone pairs) is the most widely used and validated. Methanol (2B: 1 OH
group) and glycols (4C: 2 OH groups) are the other common scheme choices.
The 1A scheme is used for carboxylic acids that form dimers.

---
## Exercise 6.3 — Compensating Parameters

**Problem:** Show graphically that high $\varepsilon$ / low $\beta$
can give the same $X_A$ as low $\varepsilon$ / high $\beta$.

### Solution

In [5]:
# Iso-XA contours in eps-beta space
rho_l = 55000.0
T_val = 373.15
b_v = 1.45e-5

eps_r = np.linspace(500, 3500, 100)
beta_r = np.linspace(0.01, 0.15, 100)
E, B = np.meshgrid(eps_r, beta_r)
XA_grid = np.zeros_like(E)

for i in range(len(beta_r)):
    for j in range(len(eps_r)):
        BF = np.exp(eps_r[j] / T_val) - 1.0
        eta = b_v * rho_l / 4.0
        g = 1.0 / (1 - 1.9 * eta)
        D = g * BF * b_v * beta_r[i]
        XA_grid[i, j] = (-1 + np.sqrt(1 + 8 * rho_l * D)) / (4 * rho_l * D)

fig, ax = plt.subplots(figsize=(3.5, 3.0))
cs = ax.contour(E, B, XA_grid, levels=[0.02, 0.05, 0.1, 0.2, 0.3, 0.5],
                colors=[BLUE, ORANGE, GREEN, PURPLE, PINK, LIME], linewidths=1.0)
ax.clabel(cs, inline=True, fontsize=7, fmt="$X_A$=%.2f")
ax.plot(2003.1, 0.0692, 'k*', markersize=10, label="Water (4C)")
ax.plot(2957.6, 0.0161, 'r^', markersize=8, label="Methanol (2B)")
ax.set_xlabel(r"$\varepsilon / R$ (K)"); ax.set_ylabel(r"$\beta$")
ax.set_title("Exercise 6.3: Iso-$X_A$ contours")
ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch06_ex03_compensating_params.png")

Saved: fig_ch06_ex03_compensating_params.png


**Answer:** The contour plot clearly shows that equal $X_A$ values trace
curves in ($\varepsilon/R$, $\beta$) space. A high association energy with
low volume can produce the same degree of bonding as a low energy with
high volume. This parameter compensation is a fundamental challenge in
CPA parameter regression and motivates using multiple data types.